# Use Case 02: Customer Support Ticket Triage

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/genai/usecases/notebooks/02-support-ticket-triage.ipynb)

**CareerAlign GenAI Use Cases**

Build an end-to-end LLM pipeline that automatically classifies support tickets by urgency and category, drafts responses using RAG over a knowledge base of resolved tickets, and routes each ticket to the correct team.

### What You'll Build
1. **50 synthetic support tickets** with ground-truth labels
2. **LLM classification pipeline** with structured JSON output (urgency P1-P4, category, sentiment)
3. **Knowledge base** from resolved tickets stored in ChromaDB
4. **RAG-powered response drafting** using similar resolved tickets as context
5. **Routing engine** with skill matching and escalation detection
6. **Evaluation** with accuracy, confusion matrix, and classification report

---
## 1. Setup & Installation

In [ ]:
!pip install -q openai chromadb pandas scikit-learn

In [ ]:
import os
import json
import re
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

# Set your OpenAI API key
# In Colab: use Secrets (key icon in sidebar) or set directly
# os.environ["OPENAI_API_KEY"] = "sk-..."

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    if "OPENAI_API_KEY" not in os.environ:
        print("WARNING: Set OPENAI_API_KEY before running LLM cells.")
    else:
        print("API key found in environment.")

In [ ]:
from openai import OpenAI
import chromadb

client = OpenAI()
print("OpenAI client initialized.")

---
## 2. Synthetic Support Tickets

We create 50 realistic support tickets spanning 5 categories and 4 urgency levels. Each ticket has ground-truth labels so we can evaluate the LLM classifier.

In [ ]:
TICKETS = [
    # === BILLING (10 tickets) ===
    {
        "id": "T001",
        "text": "I was charged twice for my monthly subscription. My card shows two charges of $49.99 on March 1st. I need a refund for the duplicate charge immediately.",
        "true_urgency": "P2",
        "true_category": "billing",
        "true_sentiment": "frustrated"
    },
    {
        "id": "T002",
        "text": "Can you explain what the $9.99 charge on my statement is for? I don't recognize it. My account is john.doe@email.com.",
        "true_urgency": "P3",
        "true_category": "billing",
        "true_sentiment": "neutral"
    },
    {
        "id": "T003",
        "text": "I cancelled my subscription two months ago but I'm STILL being charged! This is fraud! I want all charges reversed and I'm filing a chargeback with my bank if this isn't resolved today.",
        "true_urgency": "P1",
        "true_category": "billing",
        "true_sentiment": "angry"
    },
    {
        "id": "T004",
        "text": "I'd like to upgrade from the Basic plan to the Pro plan. Can you tell me if I'll be charged the full price or prorated for the rest of the month?",
        "true_urgency": "P4",
        "true_category": "billing",
        "true_sentiment": "neutral"
    },
    {
        "id": "T005",
        "text": "My invoice for February shows a charge for 15 users but we only have 10 active users. Something is wrong with the billing. We're a small startup and can't afford to overpay.",
        "true_urgency": "P2",
        "true_category": "billing",
        "true_sentiment": "frustrated"
    },
    {
        "id": "T006",
        "text": "Do you offer annual billing? I'd like to switch from monthly to annual if there's a discount.",
        "true_urgency": "P4",
        "true_category": "billing",
        "true_sentiment": "positive"
    },
    {
        "id": "T007",
        "text": "Payment failed when I tried to renew. My card is valid and has funds. I can't use the service until payment goes through. Error code: PAY_DECLINED_001.",
        "true_urgency": "P2",
        "true_category": "billing",
        "true_sentiment": "frustrated"
    },
    {
        "id": "T008",
        "text": "I need a copy of my receipt for expense reporting. Could you send me the invoice for January 2026?",
        "true_urgency": "P4",
        "true_category": "billing",
        "true_sentiment": "neutral"
    },
    {
        "id": "T009",
        "text": "We signed a contract for enterprise pricing at $5/user/month but I'm being billed at the standard rate of $12/user. This needs to be corrected immediately for all 200 users.",
        "true_urgency": "P1",
        "true_category": "billing",
        "true_sentiment": "angry"
    },
    {
        "id": "T010",
        "text": "Thanks for the quick resolution on my refund request last week! I just wanted to confirm the credit showed up on my statement. You guys have great support.",
        "true_urgency": "P4",
        "true_category": "billing",
        "true_sentiment": "positive"
    },

    # === TECHNICAL (12 tickets) ===
    {
        "id": "T011",
        "text": "The entire platform is down. None of our 500 users can access the dashboard. We're getting a 503 error on every page. This is impacting our business operations.",
        "true_urgency": "P1",
        "true_category": "technical",
        "true_sentiment": "angry"
    },
    {
        "id": "T012",
        "text": "The export to CSV feature is broken. When I click export, it downloads an empty file. This has been happening since the update last Tuesday.",
        "true_urgency": "P2",
        "true_category": "technical",
        "true_sentiment": "frustrated"
    },
    {
        "id": "T013",
        "text": "I found a typo on the settings page. The word 'preferences' is spelled 'preferances'. Minor thing but thought you'd want to know.",
        "true_urgency": "P4",
        "true_category": "technical",
        "true_sentiment": "neutral"
    },
    {
        "id": "T014",
        "text": "The API is returning 500 errors intermittently. About 10% of our requests fail. We've been seeing this for 3 days. Here's a sample error response: {\"error\": \"internal_server_error\", \"request_id\": \"abc123\"}.",
        "true_urgency": "P1",
        "true_category": "technical",
        "true_sentiment": "frustrated"
    },
    {
        "id": "T015",
        "text": "How do I set up single sign-on (SSO) with our Azure Active Directory? I looked at the docs but couldn't find step-by-step instructions.",
        "true_urgency": "P3",
        "true_category": "technical",
        "true_sentiment": "neutral"
    },
    {
        "id": "T016",
        "text": "Dashboard is extremely slow. Pages take 15-20 seconds to load. It used to be instant. I've cleared my cache and tried different browsers. Same issue on Chrome, Firefox, and Safari.",
        "true_urgency": "P2",
        "true_category": "technical",
        "true_sentiment": "frustrated"
    },
    {
        "id": "T017",
        "text": "When I upload a file larger than 50MB, the upload hangs at 99% and then fails. Smaller files work fine. I need to upload a 200MB dataset for my project.",
        "true_urgency": "P3",
        "true_category": "technical",
        "true_sentiment": "neutral"
    },
    {
        "id": "T018",
        "text": "CRITICAL: We just discovered that our data from the last 24 hours appears to be missing from the database. All entries after March 5th 2pm UTC are gone. This is a potential data loss incident.",
        "true_urgency": "P1",
        "true_category": "technical",
        "true_sentiment": "angry"
    },
    {
        "id": "T019",
        "text": "The mobile app crashes every time I try to open the reports section. I'm on iOS 18.3, iPhone 16 Pro. App version 4.2.1.",
        "true_urgency": "P2",
        "true_category": "technical",
        "true_sentiment": "frustrated"
    },
    {
        "id": "T020",
        "text": "Is there a way to connect your API with our Salesforce instance? Looking for documentation on the integration.",
        "true_urgency": "P3",
        "true_category": "technical",
        "true_sentiment": "neutral"
    },
    {
        "id": "T021",
        "text": "The search feature returns no results for queries that definitely have matches. For example, searching for 'Q4 report' returns nothing even though I uploaded that document yesterday.",
        "true_urgency": "P2",
        "true_category": "technical",
        "true_sentiment": "frustrated"
    },
    {
        "id": "T022",
        "text": "Security concern: I received an email that looks like it's from your platform asking me to reset my password, but the link goes to a suspicious domain. Was this from you?",
        "true_urgency": "P1",
        "true_category": "technical",
        "true_sentiment": "neutral"
    },

    # === ACCOUNT (10 tickets) ===
    {
        "id": "T023",
        "text": "I can't log into my account. I've tried resetting my password three times but never receive the reset email. I need access urgently for a client presentation in 2 hours.",
        "true_urgency": "P1",
        "true_category": "account",
        "true_sentiment": "frustrated"
    },
    {
        "id": "T024",
        "text": "Please update the email address on my account from old@company.com to new@company.com. I recently changed roles.",
        "true_urgency": "P3",
        "true_category": "account",
        "true_sentiment": "neutral"
    },
    {
        "id": "T025",
        "text": "I need to add 5 new team members to our organization account. Their emails are: alice@co.com, bob@co.com, carol@co.com, dave@co.com, eve@co.com.",
        "true_urgency": "P3",
        "true_category": "account",
        "true_sentiment": "neutral"
    },
    {
        "id": "T026",
        "text": "Someone has accessed my account without authorization. I can see login activity from an IP address in a country I've never been to. Please lock my account immediately and investigate.",
        "true_urgency": "P1",
        "true_category": "account",
        "true_sentiment": "angry"
    },
    {
        "id": "T027",
        "text": "How do I enable two-factor authentication on my account? I want to improve security.",
        "true_urgency": "P3",
        "true_category": "account",
        "true_sentiment": "neutral"
    },
    {
        "id": "T028",
        "text": "I want to delete my account and all associated data. Please confirm that all my data will be permanently removed in compliance with GDPR.",
        "true_urgency": "P2",
        "true_category": "account",
        "true_sentiment": "neutral"
    },
    {
        "id": "T029",
        "text": "My account has been locked after too many failed login attempts. I know my password is correct. The error says 'account suspended due to security policy'. Please unlock it.",
        "true_urgency": "P2",
        "true_category": "account",
        "true_sentiment": "frustrated"
    },
    {
        "id": "T030",
        "text": "I need to transfer ownership of our organization account from my email to my colleague's email. I'm leaving the company next week.",
        "true_urgency": "P2",
        "true_category": "account",
        "true_sentiment": "neutral"
    },
    {
        "id": "T031",
        "text": "I just signed up but my account says it's in a 'pending verification' state. I never received the verification email. I've checked spam.",
        "true_urgency": "P3",
        "true_category": "account",
        "true_sentiment": "neutral"
    },
    {
        "id": "T032",
        "text": "I'm an admin and I accidentally removed myself from the admin role. Now no one in our organization has admin access. We have 50 users who need me to manage their permissions.",
        "true_urgency": "P1",
        "true_category": "account",
        "true_sentiment": "frustrated"
    },

    # === FEATURE REQUEST (10 tickets) ===
    {
        "id": "T033",
        "text": "It would be great if the dashboard had a dark mode option. I work late nights and the bright white interface is hard on my eyes.",
        "true_urgency": "P4",
        "true_category": "feature_request",
        "true_sentiment": "positive"
    },
    {
        "id": "T034",
        "text": "Can you add the ability to schedule reports to be emailed automatically? I currently have to manually export and send them every Monday.",
        "true_urgency": "P4",
        "true_category": "feature_request",
        "true_sentiment": "neutral"
    },
    {
        "id": "T035",
        "text": "We desperately need a bulk import feature. Adding items one by one when we have 10,000 records is not feasible. This is a blocker for our team adopting the platform.",
        "true_urgency": "P3",
        "true_category": "feature_request",
        "true_sentiment": "frustrated"
    },
    {
        "id": "T036",
        "text": "Would love to see Slack integration so we get notifications in our team channel when important events happen.",
        "true_urgency": "P4",
        "true_category": "feature_request",
        "true_sentiment": "positive"
    },
    {
        "id": "T037",
        "text": "Please add keyboard shortcuts for common actions. I use the platform 8 hours a day and having to click through menus for everything slows me down significantly.",
        "true_urgency": "P4",
        "true_category": "feature_request",
        "true_sentiment": "neutral"
    },
    {
        "id": "T038",
        "text": "Is there a roadmap for adding multi-language support? We're a global company and our French and German teams can't use the platform because it's English-only.",
        "true_urgency": "P3",
        "true_category": "feature_request",
        "true_sentiment": "neutral"
    },
    {
        "id": "T039",
        "text": "The mobile app is missing the analytics section entirely. I can only see analytics on desktop. Any plans to add this to mobile?",
        "true_urgency": "P4",
        "true_category": "feature_request",
        "true_sentiment": "neutral"
    },
    {
        "id": "T040",
        "text": "Can we get an audit log feature? For compliance reasons, we need to track who accessed what data and when. This is becoming a requirement for our SOC 2 certification.",
        "true_urgency": "P3",
        "true_category": "feature_request",
        "true_sentiment": "neutral"
    },
    {
        "id": "T041",
        "text": "Love the product! One suggestion: it would be amazing if we could customize the dashboard widgets. Different teams need different metrics at a glance.",
        "true_urgency": "P4",
        "true_category": "feature_request",
        "true_sentiment": "positive"
    },
    {
        "id": "T042",
        "text": "Feature request: API rate limit headers. We need X-RateLimit-Remaining and X-RateLimit-Reset headers in API responses so our app can handle throttling gracefully.",
        "true_urgency": "P4",
        "true_category": "feature_request",
        "true_sentiment": "neutral"
    },

    # === GENERAL (8 tickets) ===
    {
        "id": "T043",
        "text": "What are your support hours? I tried calling on Saturday but no one answered.",
        "true_urgency": "P4",
        "true_category": "general",
        "true_sentiment": "neutral"
    },
    {
        "id": "T044",
        "text": "I'm evaluating your product for our team of 200 people. Can you put me in touch with someone from sales? We have specific questions about enterprise features.",
        "true_urgency": "P3",
        "true_category": "general",
        "true_sentiment": "positive"
    },
    {
        "id": "T045",
        "text": "Do you have any case studies or whitepapers about how other companies in the healthcare industry use your platform?",
        "true_urgency": "P4",
        "true_category": "general",
        "true_sentiment": "neutral"
    },
    {
        "id": "T046",
        "text": "This is the worst service I have ever experienced. I've been waiting 5 days for a response to my previous ticket. I'm going to post about this on social media and write a review. I want to speak to a manager immediately.",
        "true_urgency": "P1",
        "true_category": "general",
        "true_sentiment": "angry"
    },
    {
        "id": "T047",
        "text": "Just wanted to say thank you to your support agent Sarah who helped me yesterday. She was incredibly patient and knowledgeable. Please pass on my appreciation.",
        "true_urgency": "P4",
        "true_category": "general",
        "true_sentiment": "positive"
    },
    {
        "id": "T048",
        "text": "Where can I find your API documentation? The link on your website seems to be broken.",
        "true_urgency": "P3",
        "true_category": "general",
        "true_sentiment": "neutral"
    },
    {
        "id": "T049",
        "text": "Is your platform HIPAA compliant? We're a healthcare provider and need to ensure all patient data is handled according to HIPAA regulations before we can proceed.",
        "true_urgency": "P3",
        "true_category": "general",
        "true_sentiment": "neutral"
    },
    {
        "id": "T050",
        "text": "I accidentally sent sensitive patient data through your platform chat. We need confirmation that this data can be permanently deleted from your servers for HIPAA compliance. This is urgent.",
        "true_urgency": "P1",
        "true_category": "general",
        "true_sentiment": "frustrated"
    },
]

df = pd.DataFrame(TICKETS)
print(f"Total tickets: {len(df)}")
print(f"\nCategory distribution:\n{df['true_category'].value_counts().to_string()}")
print(f"\nUrgency distribution:\n{df['true_urgency'].value_counts().to_string()}")
print(f"\nSentiment distribution:\n{df['true_sentiment'].value_counts().to_string()}")

---
## 3. Ticket Preprocessing

Clean raw ticket text: strip HTML, remove signatures, mask PII.

In [ ]:
def preprocess_ticket(raw_text: str, channel: str = "email") -> dict:
    """Clean and normalize a support ticket."""
    text = re.sub(r'<[^>]+>', '', raw_text)             # strip HTML
    text = re.sub(r'--\s*\n[\s\S]*$', '', text)        # remove email sigs
    text = re.sub(r'Sent from my [\w\s]+$', '', text)
    text = re.sub(r'^>.*$', '', text, flags=re.MULTILINE)  # remove quoted replies
    text = re.sub(r'\s+', ' ', text).strip()
    text = text[:2000]  # truncate
    return {
        "cleaned_text": text,
        "channel": channel,
        "word_count": len(text.split()),
    }


def mask_pii(text: str) -> str:
    """Mask common PII patterns."""
    text = re.sub(r'\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b', '[CREDIT_CARD]', text)
    text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[SSN]', text)
    text = re.sub(r'\b[\w.-]+@[\w.-]+\.\w+\b', '[EMAIL]', text)
    text = re.sub(r'\b\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}\b', '[PHONE]', text)
    return text


# Test on a sample
sample = TICKETS[1]["text"]
print("Original:", sample)
processed = preprocess_ticket(sample)
masked = mask_pii(processed["cleaned_text"])
print("\nPreprocessed:", processed)
print("\nPII-masked:", masked)

---
## 4. LLM Classification with Structured Output

We use OpenAI's `response_format` with `json_schema` to force the model to return valid, structured JSON for every ticket classification.

In [ ]:
CLASSIFICATION_SCHEMA = {
    "type": "object",
    "properties": {
        "urgency": {
            "type": "string",
            "enum": ["P1", "P2", "P3", "P4"],
            "description": "P1=critical outage/data loss/security, P2=major feature broken, P3=minor bug/how-to, P4=feature request/feedback/low-impact"
        },
        "category": {
            "type": "string",
            "enum": ["billing", "technical", "account", "feature_request", "general"]
        },
        "subcategory": {"type": "string"},
        "sentiment": {
            "type": "string",
            "enum": ["angry", "frustrated", "neutral", "positive"]
        },
        "summary": {"type": "string"},
        "escalation_risk": {"type": "boolean"}
    },
    "required": ["urgency", "category", "subcategory", "sentiment", "summary", "escalation_risk"],
    "additionalProperties": False
}

SYSTEM_PROMPT = """You are a support ticket classifier for a SaaS company. Analyze each ticket and classify it.

Priority guidelines:
- P1: Service outage, data loss, security breach, complete inability to use product, unauthorized access
- P2: Major feature broken, significant performance degradation, billing errors with financial impact, account locked
- P3: Minor bug, how-to question, general inquiry, account changes, compliance questions
- P4: Feature request, feedback, cosmetic issues, informational requests

Category guidelines:
- billing: Charges, invoices, refunds, pricing, payment issues, subscription changes
- technical: Bugs, errors, performance, API issues, integrations, security vulnerabilities
- account: Login, password, access, permissions, account settings, user management
- feature_request: New features, improvements, product suggestions, roadmap questions
- general: Support hours, sales inquiries, documentation, compliance, praise, complaints about service

Set escalation_risk to true if the customer mentions: cancellation, legal action, social media complaints, chargeback, or expresses extreme frustration."""


def classify_ticket(ticket_text: str) -> dict:
    """Classify a ticket using structured output."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": ticket_text},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "ticket_classification",
                "strict": True,
                "schema": CLASSIFICATION_SCHEMA,
            },
        },
        temperature=0.0,
    )
    return json.loads(response.choices[0].message.content)


# Test on a single ticket
test_result = classify_ticket(TICKETS[0]["text"])
print("Ticket:", TICKETS[0]["text"][:80], "...")
print("\nClassification:")
print(json.dumps(test_result, indent=2))

---
## 5. Classify All 50 Tickets

Run the classifier on every ticket and compare against ground-truth labels.

In [ ]:
import time

results = []

print("Classifying 50 tickets...\n")
for i, ticket in enumerate(TICKETS):
    cleaned = mask_pii(preprocess_ticket(ticket["text"])["cleaned_text"])
    classification = classify_ticket(cleaned)

    results.append({
        "id": ticket["id"],
        "text": ticket["text"][:60] + "...",
        "true_urgency": ticket["true_urgency"],
        "pred_urgency": classification["urgency"],
        "true_category": ticket["true_category"],
        "pred_category": classification["category"],
        "true_sentiment": ticket["true_sentiment"],
        "pred_sentiment": classification["sentiment"],
        "summary": classification["summary"],
        "escalation_risk": classification["escalation_risk"],
    })

    # Progress indicator
    if (i + 1) % 10 == 0:
        print(f"  Classified {i+1}/50 tickets")

    time.sleep(0.1)  # Rate-limit courtesy

results_df = pd.DataFrame(results)
print("\nDone! Preview:")
results_df[["id", "true_category", "pred_category", "true_urgency", "pred_urgency"]].head(10)

---
## 6. Evaluation: Accuracy, Confusion Matrix & Classification Report

In [ ]:
# ── Category Accuracy ──
cat_acc = accuracy_score(results_df["true_category"], results_df["pred_category"])
print(f"Category Accuracy: {cat_acc:.1%}")
print("\nCategory Classification Report:")
print(classification_report(
    results_df["true_category"],
    results_df["pred_category"],
    zero_division=0
))

# ── Urgency Accuracy ──
urg_acc = accuracy_score(results_df["true_urgency"], results_df["pred_urgency"])
print(f"\nUrgency Accuracy: {urg_acc:.1%}")
print("\nUrgency Classification Report:")
print(classification_report(
    results_df["true_urgency"],
    results_df["pred_urgency"],
    zero_division=0
))

# ── Sentiment Accuracy ──
sent_acc = accuracy_score(results_df["true_sentiment"], results_df["pred_sentiment"])
print(f"\nSentiment Accuracy: {sent_acc:.1%}")
print("\nSentiment Classification Report:")
print(classification_report(
    results_df["true_sentiment"],
    results_df["pred_sentiment"],
    zero_division=0
))

In [ ]:
# ── Confusion Matrices (text-based, no matplotlib needed) ──

def print_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    """Print a text-based confusion matrix."""
    labels = sorted(set(list(y_true) + list(y_pred)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print(f"\n{title}")
    print("Predicted →")
    print(f"{'Actual ↓':<18}", "  ".join(f"{l:>12}" for l in labels))
    print("-" * (18 + 14 * len(labels)))
    for i, label in enumerate(labels):
        row = "  ".join(f"{v:>12}" for v in cm[i])
        print(f"{label:<18} {row}")

print_confusion_matrix(
    results_df["true_category"], results_df["pred_category"],
    "Category Confusion Matrix"
)

print_confusion_matrix(
    results_df["true_urgency"], results_df["pred_urgency"],
    "Urgency Confusion Matrix"
)

In [ ]:
# ── Escalation Detection Analysis ──

# Tickets that should trigger escalation (angry sentiment or P1)
should_escalate = (
    (results_df["true_sentiment"] == "angry") |
    (results_df["true_urgency"] == "P1")
)
predicted_escalation = results_df["escalation_risk"]

escalation_recall = (
    (should_escalate & predicted_escalation).sum() / should_escalate.sum()
    if should_escalate.sum() > 0 else 0
)
escalation_precision = (
    (should_escalate & predicted_escalation).sum() / predicted_escalation.sum()
    if predicted_escalation.sum() > 0 else 0
)

print(f"Escalation Detection:")
print(f"  Tickets that should escalate: {should_escalate.sum()}")
print(f"  Tickets flagged for escalation: {predicted_escalation.sum()}")
print(f"  Recall: {escalation_recall:.1%}")
print(f"  Precision: {escalation_precision:.1%}")

---
## 7. Knowledge Base & RAG Response Drafting

Build a ChromaDB vector store from synthetic "resolved" tickets, then use RAG to draft responses for new tickets.

In [ ]:
# Synthetic resolved tickets — our knowledge base
RESOLVED_TICKETS = [
    {
        "id": "R001",
        "text": "Customer was charged twice for monthly subscription due to a payment processing glitch.",
        "category": "billing",
        "resolution": "Issued a full refund for the duplicate charge. Confirmed with payment team that the glitch has been patched.",
        "response": "I sincerely apologize for the duplicate charge. I've processed a full refund which should appear on your statement within 3-5 business days. Our team has identified and fixed the processing issue that caused this."
    },
    {
        "id": "R002",
        "text": "Customer asking about an unrecognized $9.99 charge on their statement.",
        "category": "billing",
        "resolution": "The charge was for the add-on feature the customer enabled last month. Explained the charge and offered to disable if unwanted.",
        "response": "Thanks for reaching out! The $9.99 charge is for the Advanced Analytics add-on that was enabled on your account on Feb 15. If you'd like to keep it, no action is needed. If you'd like to disable it, I can do that right away and issue a prorated refund."
    },
    {
        "id": "R003",
        "text": "Customer still being charged after cancelling subscription two months ago.",
        "category": "billing",
        "resolution": "Cancellation was not processed correctly due to a system error. Manually cancelled and refunded all charges since cancellation date.",
        "response": "I'm very sorry about this. I can see your cancellation request was submitted but wasn't processed due to a system error on our end. I've now fully cancelled your subscription and processed refunds for all charges since your original cancellation date. You'll see the credits within 5-7 business days."
    },
    {
        "id": "R004",
        "text": "Customer cannot log in after multiple failed attempts. Account is locked.",
        "category": "account",
        "resolution": "Verified customer identity via security questions. Unlocked account and sent password reset link.",
        "response": "I've verified your identity and unlocked your account. I've also sent a password reset link to your registered email. Please use it to set a new password within the next 24 hours. If you don't see the email, please check your spam folder."
    },
    {
        "id": "R005",
        "text": "Customer reports unauthorized access to their account from unknown IP address.",
        "category": "account",
        "resolution": "Immediately locked account, invalidated all sessions, forced password reset. Escalated to security team for investigation.",
        "response": "Thank you for reporting this immediately. I've locked your account and invalidated all active sessions as a precaution. Our security team is investigating the unauthorized access. I'll send you a secure password reset link shortly. I strongly recommend enabling two-factor authentication once you regain access."
    },
    {
        "id": "R006",
        "text": "Platform returning 503 errors. All users affected. Service outage.",
        "category": "technical",
        "resolution": "Infrastructure team identified a database connection pool exhaustion. Scaled up connection pool and restarted affected services. Added monitoring alerts.",
        "response": "We've identified and resolved the issue that caused the service outage. Our infrastructure team found that a database connection pool was exhausted, causing 503 errors. The service has been fully restored and we've added additional monitoring to prevent this from recurring. We apologize for the disruption."
    },
    {
        "id": "R007",
        "text": "CSV export downloads empty file. Bug introduced in recent update.",
        "category": "technical",
        "resolution": "Bug confirmed in v4.2.0 release. Hotfix deployed to fix the CSV export encoding issue.",
        "response": "Thank you for reporting this. We've confirmed a bug in our latest release that affected CSV exports. A fix has been deployed and exports should now work correctly. Please try again and let us know if you encounter any further issues."
    },
    {
        "id": "R008",
        "text": "Dashboard extremely slow, pages take 15-20 seconds to load.",
        "category": "technical",
        "resolution": "Identified a slow database query caused by missing index on the user_activity table. Added index and response time dropped to under 1 second.",
        "response": "We've identified the cause of the slow performance. A database optimization was needed for the dashboard queries. We've applied the fix and you should see significantly faster load times now. Please refresh your browser and let us know if the issue persists."
    },
    {
        "id": "R009",
        "text": "Customer wants to enable two-factor authentication.",
        "category": "account",
        "resolution": "Provided step-by-step instructions for enabling 2FA via Settings > Security > Two-Factor Authentication.",
        "response": "Great decision to enable 2FA! Here's how: Go to Settings > Security > Two-Factor Authentication > Enable. You can use any authenticator app (Google Authenticator, Authy, etc.). Scan the QR code shown on screen, enter the 6-digit code, and save your backup codes in a secure location."
    },
    {
        "id": "R010",
        "text": "Customer requesting dark mode feature for the dashboard.",
        "category": "feature_request",
        "resolution": "Logged as feature request FR-2847. Dark mode is on the Q3 roadmap.",
        "response": "Thanks for the suggestion! Dark mode is something our team has been planning. It's currently on our roadmap for Q3 this year. I've logged your request to help prioritize this feature. We'll notify you when it becomes available."
    },
    {
        "id": "R011",
        "text": "Customer asking about HIPAA compliance status for healthcare use.",
        "category": "general",
        "resolution": "Confirmed HIPAA compliance. Shared BAA template and compliance documentation.",
        "response": "Yes, our platform is fully HIPAA compliant. We can provide a Business Associate Agreement (BAA) for your organization. I'll send over our compliance documentation and BAA template. Our enterprise plan includes all the security controls required for healthcare data."
    },
    {
        "id": "R012",
        "text": "Payment failed with error PAY_DECLINED_001 when trying to renew subscription.",
        "category": "billing",
        "resolution": "Error was due to expired card on file. Helped customer update payment method and processed renewal.",
        "response": "The PAY_DECLINED_001 error typically occurs when the card on file has expired or the bank has declined the transaction. Please try updating your payment method in Settings > Billing > Payment Methods. If the issue persists after updating, please contact your bank to ensure they're not blocking the charge."
    },
    {
        "id": "R013",
        "text": "API returning intermittent 500 errors affecting about 10% of requests.",
        "category": "technical",
        "resolution": "Root cause was a race condition in the API gateway under high load. Deployed fix with request queuing.",
        "response": "We've identified and fixed the root cause of the intermittent 500 errors. A race condition in our API gateway was causing failures under high request volume. We've deployed a fix that includes request queuing to handle load spikes gracefully. Please monitor your error rates and let us know if you see any further issues."
    },
    {
        "id": "R014",
        "text": "Enterprise customer being billed at standard rate instead of contracted rate.",
        "category": "billing",
        "resolution": "Contract pricing not applied due to provisioning error. Corrected pricing and issued credit for difference.",
        "response": "I sincerely apologize for the billing discrepancy. I've corrected your account to reflect the contracted rate of $5/user/month, and I'm processing a credit for the difference you were overcharged. The credit will appear on your next invoice. I've also flagged this with our provisioning team to prevent similar issues."
    },
    {
        "id": "R015",
        "text": "Mobile app crashes on iOS when opening reports section.",
        "category": "technical",
        "resolution": "Crash traced to memory leak in charts rendering library on iOS 18. Hotfix released in app version 4.2.2.",
        "response": "Thank you for the detailed report including your device and app version. We've identified a memory issue specific to iOS 18 in the reports section. A fix has been released in app version 4.2.2. Please update the app from the App Store and the crashes should be resolved."
    },
]

print(f"Knowledge base: {len(RESOLVED_TICKETS)} resolved tickets")
print(f"Categories: {Counter(t['category'] for t in RESOLVED_TICKETS)}")

In [ ]:
# ── Build ChromaDB Knowledge Base ──

chroma_client = chromadb.Client()

# Delete collection if it exists (for re-runs)
try:
    chroma_client.delete_collection("resolved_tickets")
except:
    pass

collection = chroma_client.create_collection(
    name="resolved_tickets",
    metadata={"hnsw:space": "cosine"}
)

# Add resolved tickets to the vector store
collection.add(
    documents=[t["text"] for t in RESOLVED_TICKETS],
    metadatas=[
        {
            "category": t["category"],
            "resolution": t["resolution"],
            "response": t["response"],
        }
        for t in RESOLVED_TICKETS
    ],
    ids=[t["id"] for t in RESOLVED_TICKETS],
)

print(f"Indexed {collection.count()} resolved tickets in ChromaDB.")

In [ ]:
# ── RAG Retrieval ──

def retrieve_similar_tickets(
    ticket_text: str,
    category: str = None,
    n_results: int = 3
) -> list[dict]:
    """Retrieve similar resolved tickets from ChromaDB."""
    query_params = {
        "query_texts": [ticket_text],
        "n_results": n_results,
    }
    if category:
        query_params["where"] = {"category": category}

    results = collection.query(**query_params)

    similar = []
    for i in range(len(results["documents"][0])):
        similar.append({
            "id": results["ids"][0][i],
            "ticket_text": results["documents"][0][i],
            "resolution": results["metadatas"][0][i]["resolution"],
            "response": results["metadatas"][0][i]["response"],
            "distance": results["distances"][0][i],
        })
    return similar


# Test retrieval
test_ticket = "I was charged twice on my credit card for the same month."
similar = retrieve_similar_tickets(test_ticket, category="billing", n_results=3)

print(f"Query: '{test_ticket}'\n")
for s in similar:
    print(f"  [{s['id']}] (distance={s['distance']:.3f}) {s['ticket_text'][:80]}...")
    print(f"  Resolution: {s['resolution'][:80]}...\n")

In [ ]:
# ── RAG Response Drafting ──

def draft_response(
    ticket_text: str,
    classification: dict,
    similar_tickets: list[dict],
) -> str:
    """Draft a response using RAG context from similar resolved tickets."""
    context = "\n\n".join([
        f"Similar Ticket: {t['ticket_text']}\n"
        f"Resolution: {t['resolution']}\n"
        f"Response Sent: {t['response']}"
        for t in similar_tickets
    ])

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": f"""You are a support agent drafting a response to a customer ticket.
Use the context from similar resolved tickets to craft a helpful, empathetic response.
Match the tone to the customer's sentiment:
- Angry/frustrated: Extra empathy, acknowledge the issue, apologize
- Neutral: Professional and efficient
- Positive: Warm and appreciative

Ticket Category: {classification.get('category', 'unknown')}
Urgency: {classification.get('urgency', 'unknown')}
Sentiment: {classification.get('sentiment', 'unknown')}

CONTEXT FROM SIMILAR RESOLVED TICKETS:
{context}

Draft a concise, professional response (2-4 sentences). Do NOT make up specific details that aren't supported by the context.""",
            },
            {
                "role": "user",
                "content": f"Draft a response for this ticket:\n\n{ticket_text}",
            },
        ],
        temperature=0.3,
    )
    return response.choices[0].message.content


# Test full RAG pipeline on a few tickets
demo_indices = [0, 10, 22, 45]  # billing, technical, account, general

for idx in demo_indices:
    ticket = TICKETS[idx]
    print(f"{'='*80}")
    print(f"Ticket {ticket['id']}: {ticket['text'][:100]}...")

    # Classify
    cls = classify_ticket(ticket["text"])
    print(f"\nClassification: {cls['urgency']} | {cls['category']} | {cls['sentiment']}")

    # Retrieve
    similar = retrieve_similar_tickets(ticket["text"], category=cls["category"])
    print(f"Retrieved {len(similar)} similar tickets")

    # Draft
    draft = draft_response(ticket["text"], cls, similar)
    print(f"\nDraft Response:\n{draft}\n")

---
## 8. Routing Engine

Map classifications to team assignments with SLA tracking and escalation rules.

In [ ]:
# ── Routing Configuration ──

ROUTING_CONFIG = {
    "billing": {
        "team": "billing_support",
        "escalation_team": "billing_lead",
        "sla_hours": {"P1": 1, "P2": 4, "P3": 24, "P4": 72},
    },
    "technical": {
        "team": "tech_support",
        "escalation_team": "engineering_oncall",
        "sla_hours": {"P1": 0.5, "P2": 2, "P3": 24, "P4": 72},
    },
    "account": {
        "team": "account_support",
        "escalation_team": "account_manager",
        "sla_hours": {"P1": 1, "P2": 4, "P3": 24, "P4": 72},
    },
    "feature_request": {
        "team": "product_feedback",
        "escalation_team": "product_manager",
        "sla_hours": {"P1": 24, "P2": 48, "P3": 72, "P4": 168},
    },
    "general": {
        "team": "general_support",
        "escalation_team": "support_lead",
        "sla_hours": {"P1": 2, "P2": 8, "P3": 24, "P4": 72},
    },
}

# ── Escalation Detection ──

ESCALATION_SIGNALS = [
    r"\bcancel\b.*\b(account|subscription|service)\b",
    r"\blawyer\b|\blegal\b|\bsue\b|\blawsuit\b",
    r"\b(social media|twitter|reddit|review)\b.*\b(post|share|write)\b",
    r"\b(worst|terrible|horrible|unacceptable)\b.*\b(experience|service)\b",
    r"\bbetter business bureau\b|\bBBB\b",
    r"\b(refund|chargeback|dispute)\b",
    r"\bescalat(e|ion)\b|\bmanager\b|\bsupervisor\b",
    r"\b(days|weeks|months)\b.*\b(waiting|no response|ignored)\b",
]


def detect_escalation_signals(ticket_text: str) -> dict:
    """Detect explicit escalation signals."""
    text_lower = ticket_text.lower()
    triggered = [p for p in ESCALATION_SIGNALS if re.search(p, text_lower)]
    return {
        "has_escalation_signals": len(triggered) > 0,
        "signal_count": len(triggered),
        "risk_level": (
            "critical" if len(triggered) >= 3
            else "high" if len(triggered) >= 2
            else "medium" if len(triggered) == 1
            else "low"
        ),
    }


def route_ticket(classification: dict, ticket_text: str) -> dict:
    """Route a classified ticket to the appropriate team."""
    category = classification["category"]
    urgency = classification["urgency"]
    config = ROUTING_CONFIG.get(category, ROUTING_CONFIG["general"])

    # Check escalation from multiple signals
    esc_signals = detect_escalation_signals(ticket_text)
    needs_escalation = (
        urgency == "P1"
        or classification.get("escalation_risk", False)
        or classification.get("sentiment") == "angry"
        or esc_signals["has_escalation_signals"]
    )

    team = config["escalation_team"] if needs_escalation else config["team"]
    sla = config["sla_hours"][urgency]

    return {
        "assigned_team": team,
        "sla_hours": sla,
        "escalated": needs_escalation,
        "escalation_signals": esc_signals,
        "category": category,
        "urgency": urgency,
    }


# Test routing on all classified tickets
print(f"{'ID':<6} {'Category':<16} {'Urgency':<8} {'Team':<22} {'SLA':<6} {'Escalated'}")
print("-" * 80)

routing_results = []
for i, row in results_df.iterrows():
    cls = {
        "category": row["pred_category"],
        "urgency": row["pred_urgency"],
        "sentiment": row["pred_sentiment"],
        "escalation_risk": row["escalation_risk"],
    }
    routing = route_ticket(cls, TICKETS[i]["text"])
    routing_results.append(routing)

    print(f"{row['id']:<6} {row['pred_category']:<16} {row['pred_urgency']:<8} "
          f"{routing['assigned_team']:<22} {routing['sla_hours']:<6} {routing['escalated']}")

In [ ]:
# ── Routing Summary Statistics ──

routing_df = pd.DataFrame(routing_results)

print("=" * 60)
print("ROUTING SUMMARY")
print("=" * 60)
print(f"\nTeam Distribution:")
print(routing_df["assigned_team"].value_counts().to_string())

print(f"\nEscalated: {routing_df['escalated'].sum()} / {len(routing_df)} tickets")
print(f"Escalation rate: {routing_df['escalated'].mean():.1%}")

print(f"\nAverage SLA by urgency:")
for urg in ["P1", "P2", "P3", "P4"]:
    mask = routing_df["urgency"] == urg
    if mask.any():
        avg_sla = routing_df.loc[mask, "sla_hours"].mean()
        print(f"  {urg}: {avg_sla:.1f} hours (n={mask.sum()})")

---
## 9. End-to-End Pipeline Demo

Run the complete triage pipeline on a brand-new ticket to see all stages in action.

In [ ]:
def triage_ticket(raw_text: str, channel: str = "email") -> dict:
    """
    End-to-end triage pipeline:
    1. Preprocess & mask PII
    2. Classify with structured output
    3. Detect escalation signals
    4. Retrieve similar resolved tickets
    5. Draft response via RAG
    6. Route to team
    """
    # Step 1: Preprocess
    processed = preprocess_ticket(raw_text, channel)
    clean_text = mask_pii(processed["cleaned_text"])

    # Step 2: Classify
    classification = classify_ticket(clean_text)

    # Step 3: Escalation detection
    esc_signals = detect_escalation_signals(raw_text)

    # Step 4: Retrieve similar tickets
    similar = retrieve_similar_tickets(
        clean_text,
        category=classification["category"],
        n_results=3
    )

    # Step 5: Draft response
    draft = draft_response(clean_text, classification, similar)

    # Step 6: Route
    routing = route_ticket(classification, raw_text)

    return {
        "classification": classification,
        "escalation_signals": esc_signals,
        "similar_tickets": [s["id"] for s in similar],
        "draft_response": draft,
        "routing": routing,
    }


# ── Demo: New ticket that wasn't in our training set ──
new_ticket = """
I've been trying to get help for over a week now with no response. Our entire
sales team of 30 people cannot access the CRM integration since your last update.
We're losing deals every day this remains broken. If this isn't fixed by end of
day, we're cancelling our enterprise subscription and moving to a competitor.
I need to speak with a manager.
""".strip()

print("NEW TICKET:")
print(new_ticket)
print("\n" + "=" * 80)

result = triage_ticket(new_ticket)

print("\nCLASSIFICATION:")
print(json.dumps(result["classification"], indent=2))

print(f"\nESCALATION SIGNALS: {result['escalation_signals']}")

print(f"\nSIMILAR RESOLVED TICKETS: {result['similar_tickets']}")

print(f"\nROUTING:")
print(f"  Team: {result['routing']['assigned_team']}")
print(f"  SLA: {result['routing']['sla_hours']} hours")
print(f"  Escalated: {result['routing']['escalated']}")

print(f"\nDRAFT RESPONSE:")
print(result["draft_response"])

In [ ]:
# ── Demo 2: A calmer ticket ──

calm_ticket = """
Hi, I just upgraded to the Pro plan yesterday but I still see the Basic plan
features on my dashboard. The upgrade confirmation email says Pro but the
account page still shows Basic. Could you look into this? Thanks!
""".strip()

print("NEW TICKET:")
print(calm_ticket)
print("\n" + "=" * 80)

result2 = triage_ticket(calm_ticket)

print("\nCLASSIFICATION:")
print(json.dumps(result2["classification"], indent=2))

print(f"\nROUTING:")
print(f"  Team: {result2['routing']['assigned_team']}")
print(f"  SLA: {result2['routing']['sla_hours']} hours")
print(f"  Escalated: {result2['routing']['escalated']}")

print(f"\nDRAFT RESPONSE:")
print(result2["draft_response"])

---
## 10. Summary

### What We Built

| Component | Technology | Purpose |
|---|---|---|
| **Preprocessing** | Python regex | Clean text, mask PII, normalize channels |
| **Classification** | OpenAI Structured Output (JSON mode) | Urgency, category, sentiment, escalation risk |
| **Knowledge Base** | ChromaDB vector store | Embed & store resolved tickets for retrieval |
| **RAG Drafting** | ChromaDB + GPT-4o-mini | Retrieve similar tickets, draft grounded responses |
| **Routing** | Rule-based engine | Map classifications to teams with SLA & escalation |
| **Escalation** | Regex pattern matching + LLM signals | Multi-layer detection for high-risk tickets |

### Key Takeaways

1. **Structured output eliminates parsing failures** — JSON schema mode guarantees valid, complete output
2. **RAG grounds responses in real data** — draft responses based on actual past resolutions, not hallucinations
3. **Layered escalation detection** catches signals that any single method would miss
4. **Human-in-the-loop** is essential — agents review drafts before sending, catching edge cases
5. **Cost is negligible** — full pipeline costs < $100/month for 50K tickets at GPT-4o-mini pricing

### Production Next Steps

- Integrate with Zendesk/Freshdesk/Intercom via webhooks
- Add feedback loop: agent corrections improve the knowledge base
- Implement multi-language support
- Add PII detection with Microsoft Presidio or AWS Comprehend
- Monitor classification drift with weekly accuracy reports
- Fine-tune a smaller model on your company's specific ticket data